# ⚙️ [1/5] Check your Google Drive

        DiverScan-release
        │
        ├── data
        │        └── yourdata_folder
        │
        ├── projects
        │        └── yourproject_folder
        │                ├── dataset
        │                ├── optuna
        │                └── training (🟢 `training_folder`)
        │                        ├── best_epoch.txt
        │                        └── ...
        │
        └── utils


Ensure you have the `best_epoch.txt` in your training folder.

# ⚙️ [2/5] Use a GPU

See the above menu. Select `Runtime` -> `Change runtime type` -> your favorite GPU -> `Save`.

In [ ]:
# Hit left ▶️ of this cell and run it!
CUDA = "cuda:0"

# ⚙️ [3/5] Authorize this notebook

In [ ]:
# Hit left ▶️ of this cell and run it!
# Mounting Google Drive is what Colab needs to reach your files. Elsewhere
# there is nothing to mount, so the notebook carries on with local paths.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not running in Google Colab; set the folder paths below to local ones.")


# ⚙️ [4/5] Load data

In [ ]:
#@title Hit left ▶️ of this cell and run it!

import os
import sys
from pathlib import Path
import pandas as pd
import pickle
from IPython.display import display

project_folder = "/content/drive/MyDrive/DiverScan-release/projects/fruitfly" #@param {type:"string"}
training_folder_name = "training" #@param {type:"string"}

project_folder = Path(project_folder)
training_folder = str(project_folder / training_folder_name)
os.chdir(training_folder)


def read_settings_file(path):
    values = {}
    with open(path, "r") as f:
        for line in f:
            key, separator, value = line.rstrip("\n").partition(": ")
            if separator:
                values[key] = value
    return values


# The training folder records which optuna folder it was trained from.
loaded_values = read_settings_file("learning_settings.txt")
optuna_folder_name = loaded_values.get("optuna_folder_name") or Path(loaded_values["optuna_folder"]).name
optuna_folder = str(project_folder / optuna_folder_name)
os.chdir(optuna_folder)


# Read learning_settings.txt and optuna_summary.txt.
# The code to import is the snapshot 01_build_dataset copied into the project,
# whose folder name is recorded in learning_settings.txt.
loaded_values = read_settings_file("learning_settings.txt")
utils_snapshot = loaded_values.get("utils_snapshot")
if utils_snapshot:
    utils_folder = str(project_folder / utils_snapshot)
else:
    utils_folder = loaded_values["utils_folder"]
    print("No code snapshot in this project; using", utils_folder)
if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import learning_utils as _learning_utils

settings = _learning_utils.read_learning_settings("learning_settings.txt")
model_type = settings["model_type"]
dataset_folder = str(project_folder / "dataset")
task_type = settings["task_type"]
number_of_attention_branches = settings["number_of_attention_branches"]
seed = settings["seed"]
batch_size = settings["batch_size"]
learning_rate = settings["learning_rate"]
C_CE = settings["C_CE"]
C_S = settings["C_S"]
threshold_of_attention_distribution = settings["threshold_of_attention_distribution"]
threshold_of_loss_penalty = settings["threshold_of_loss_penalty"]
lambda_for_total_variation_regularization = settings["lambda_for_total_variation_regularization"]
conv1d_out_channels = settings["conv1d_out_channels"]
lstm_hidden_size = settings["lstm_hidden_size"]
stgcn_out_channels = settings["stgcn_out_channels"]
node_shuffle = settings["node_shuffle"]

optuna_values = _learning_utils.read_optuna_summary_settings(model_type, "optuna_summary.txt")
dropout_probability = optuna_values["dropout_probability"]
weight_decay = optuna_values["weight_decay"]
temporal_kernel_size_before_branching = optuna_values["temporal_kernel_size_before_branching"]
before_branching_blocks = optuna_values["before_branching_blocks"]
macro_averaged_accuracies = optuna_values["macro_averaged_accuracies"]
similarities = optuna_values["similarities"]

os.chdir(dataset_folder)   # make sure the dataset is where it should be
os.chdir(training_folder)

# Read best_epoch.txt
loaded_values = {}
with open("best_epoch.txt", "r") as f:
    for line in f:
        parts = line.strip().split(" = ")
        if len(parts) == 2:
            key, parameter = parts
            loaded_values[key] = parameter

best_epoch_for_analysis = int(loaded_values["epoch"])
macro_averaged_accuracies = [eval(loaded_values["macro_averaged_accuracies"])]
similarities = [eval(loaded_values["similarities"])]

print("\n\nUsing the validation set...")
print("\n⭐️ Epoch for analysis:", best_epoch_for_analysis, "⭐️")

df_single_row_1 = pd.DataFrame(macro_averaged_accuracies)
df_single_row_2 = pd.DataFrame(similarities)
df_combined = pd.concat([df_single_row_1, df_single_row_2], axis=0, ignore_index=True)

df_combined.columns = [f"Attention{i+1}" for i in range(len(df_combined.columns))]
df_combined.index = ["Macro averaged accuracy", "Similarity"]

display(df_combined)

#@markdown ⭐️ Here you can check the summary of `03_train_model`.

#@markdown - `macro averaged accuracy`: Higher values are better. Value 1 means perfectly successful classification, suggesting your model finds difference(s) over conditions. Value 0 means completely failed classification.
#@markdown - `similarity`: Lower values are better. Value 0 means completely different attentions across all branches, suggesting your attention branches find diverse features. Value 1 means completely identical attentions across all branches.

#@markdown ⭐️ If your result has problems...
#@markdown - The number of epochs used in `03_train_model` might not be sufficient.
#@markdown - There may be no significant differences in the comparison you conducted.


# ⚙️ [5/5] Test your model

1.   Input followings, then select `Runtime` -> `Run all`.

2.   When the connection shuts down after all executions are complete, save and then close the notebook.

3.   Go to the `05_visualize_results` notebook.

In [ ]:
#@title Advanced settings
do_not_use_the_best_epoch = False #@param {type:"boolean"}
userselect_epoch_for_analysis = 49 #@param {type:"integer"}
#@markdown - If you want to set a specific epoch of your choice for `05_visualize_results`, check `do_not_use_the_best_epoch` and enter the epoch number in `userselect_epoch_for_analysis`.


if do_not_use_the_best_epoch == False:
    epoch = best_epoch_for_analysis
else:
    epoch = userselect_epoch_for_analysis

# Codes

In [ ]:
import os

os.environ["PYTHONHASHSEED"] = str(seed)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("CUBLAS_WORKSPACE_CONFIG =", os.environ["CUBLAS_WORKSPACE_CONFIG"])

In [ ]:
from pathlib import Path
import gzip
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import copy
import gc

In [ ]:
# Make the DiverScan utils importable.
import sys

if utils_folder not in sys.path:
    sys.path.insert(0, utils_folder)

import learning_utils as learning_utils_module

if model_type == "single":
    import single_model as model_module
elif model_type == "multi":
    import multi_model as model_module
else:
    raise ValueError(f'model_type must be "single" or "multi", got {model_type!r}')

# Used for the post-analysis below.
import postprocessing as postprocessing_module

# Used to record the notebook and environment of each run.
import run_record as run_record_module


In [ ]:
# Fix the random seeds, switch on deterministic mode, and pick the device.
# https://pytorch.org/docs/stable/notes/randomness.html
DEVICE = learning_utils_module.setup_determinism(seed, CUDA)


In [ ]:
# Create a folder to save data
training_folder = Path(training_folder)
save_folder = training_folder / f"epoch{epoch}_for_visualization"

if not save_folder.exists():
    save_folder.mkdir(parents=True)
else:  # When the folder is exist
    counter = 1
    while save_folder.exists():
        save_folder = training_folder / f"epoch{epoch}_for_visualization_{counter}"
        counter += 1
    save_folder.mkdir(parents=True)

# Change current directory
os.chdir(save_folder)

In [ ]:
# Record this notebook and the environment used for this run.
# The notebook file to record, assuming the layout DiverScan/notebooks and
# DiverScan/projects/<project>. In Colab the running notebook is fetched from
# the frontend instead, so the file is only needed outside Colab.
run_record_module.save_run_record(
    utils_folder,
    training_folder,
    notebook_path=project_folder.parent.parent / "notebooks" / "04_test_model.ipynb",
)


In [ ]:
# Change current directory
os.chdir(dataset_folder)

# Load standardized data
with gzip.open("dataset_standardized_for_machine_learning.pkl", "rb") as file:
    loaded_data = pickle.load(file)
train_val_test_dictionary = loaded_data["train_val_test_dictionary"]

_, number_of_classes, number_of_channels, number_of_timestamps, number_of_nodes, number_of_dimensions_of_edge_weight, number_of_channels_of_pose, test_x, test_label, edge_index, test_edge_weight, test_x_pose, has_pose = learning_utils_module.load_data(
    "test",
    train_val_test_dictionary,
    model_type,
    batch_size,
    DEVICE,
)

In [ ]:
# Calculate some hyperparameters
threshold_of_loss_penalty, lambda_for_total_variation_regularization, penalty_weight_for_attention_distribution = learning_utils_module.calculate_hyperparameters(
    threshold_of_loss_penalty,
    lambda_for_total_variation_regularization,
    C_CE,
    C_S,
    number_of_classes,
    number_of_attention_branches,
)
model_spec = learning_utils_module.ModelSpec(
    model_type=model_type,
    number_of_classes=number_of_classes,
    number_of_channels=number_of_channels,
    number_of_timestamps=number_of_timestamps,
    number_of_attention_branches=number_of_attention_branches,
    number_of_channels_of_pose=number_of_channels_of_pose,
    number_of_nodes=number_of_nodes,
    number_of_dimensions_of_edge_weight=number_of_dimensions_of_edge_weight,
    edge_index=edge_index,
    conv1d_out_channels=conv1d_out_channels,
    lstm_hidden_size=lstm_hidden_size,
    stgcn_out_channels=stgcn_out_channels,
    node_shuffle=node_shuffle,
)


In [ ]:
# Change current directory
os.chdir(optuna_folder)

# Read branch information from optuna_summary.txt
with open("optuna_summary.txt", "r") as file:
    optuna_summary = file.read()
branch_list = learning_utils_module.read_optuna_summary(optuna_summary)

In [ ]:
# Generate configs
before_branch_config, branch_config = learning_utils_module.build_configs(
    model_spec,
    dropout_probability,
    before_branching_blocks,
    temporal_kernel_size_before_branching,
    branch_list,
)


In [ ]:
# Load the model
model = learning_utils_module.build_model(
    model_module,
    model_spec,
    before_branch_config,
    branch_config,
    DEVICE,
)

print(model)


In [ ]:
# Change current directory
os.chdir(training_folder)

# Load the parameters of the model
checkpoint_file = training_folder / f"{epoch}.pth"

if checkpoint_file.exists():
    checkpoint = torch.load(checkpoint_file)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Epoch {epoch} parameter is loaded.")
else:
    print(f'Epoch {epoch}.pth file is not found. Is "userselect_epoch_for_analysis" correct?')

In [ ]:
# Evaluation. Get predictions and attentions.
prediction_tensor_list, attentions_tensor = learning_utils_module.run_inference(
    model,
    model_type,
    test_x,
    test_edge_weight,
    test_x_pose,
    has_pose,
)


In [ ]:
# Delete large-sized data
del loaded_data
del train_val_test_dictionary
del test_x
del edge_index
del test_edge_weight
del test_x_pose

gc.collect()
torch.cuda.empty_cache()

In [ ]:
########## For visualization ##########

In [ ]:
# Change current directory
os.chdir(dataset_folder)

# Load unstandardized data
with gzip.open("dataset_unstandardized_for_post_analysis.pkl", "rb") as file:
    post_analysis_dictionary = pickle.load(file)

# Standardized data and unstandardized data should be calculated from the same original data.
if not np.array_equal(test_label.to("cpu").detach().numpy().copy(), post_analysis_dictionary["dictionary"]["onehot_labels_array"]):
    raise ValueError("The labels of the standardized and unstandardized data do not match.")

In [ ]:
# Write the accuracy tables, classification report, prediction tables,
# correlations, histograms and interactive-plot data to save_folder.
results = postprocessing_module.save_all_results(
    model_type,
    task_type,
    number_of_attention_branches,
    number_of_classes,
    has_pose,
    post_analysis_dictionary,
    prediction_tensor_list,
    attentions_tensor,
    test_label,
    save_folder,
)

attentions = results["attentions"]
pred_masks = results["pred_masks"]


In [ ]:
try:
    from google.colab import runtime
    import time
    time.sleep(5)
    runtime.unassign()
    print("The runtime has been unassigned.")
except ImportError:
    # Not running in Google Colab environment
    pass